# Stable Log-Sum-Exp, Softmax, and Cross-Entropy

## Objective
Implement numerically stable classification calculations and examine overflow, underflow, and common-shift behavior.

## Stable Formulas
Let \(m = \max_i z_i\).

$$
\operatorname{LSE}(z)
= m + \ln\left(\sum_i e^{z_i-m}\right)
$$

$$
p_i = \frac{e^{z_i-m}}{\sum_j e^{z_j-m}}
$$

For correct-class index \(y\):

$$
L = \operatorname{LSE}(z)-z_y.
$$

## Validation

| Logits | Expected LSE | Cross-entropy, target index 1 |
|---|---:|---:|
| `[1000, 1000]` | ≈1000.693147 | ≈0.693147 |
| `[0, -1000]` | ≈0 | ≈1000 |
| `[-1000, -1000]` | ≈−999.306853 | ≈0.693147 |

Adding the same constant to every logit adds that constant to LSE while leaving softmax probabilities and cross-entropy unchanged.

## Numerical Stability
Maximum subtraction prevents exponential overflow for finite logits. Computing cross-entropy directly from logits avoids taking the logarithm of a tiny probability that has rounded to zero.

## Scope
The finite-input practical passed after correcting the cross-entropy subtraction direction with guidance. It does not reproduce the paper’s full numerical error analysis.

## Reference
*Accurate Computation of the Log-Sum-Exp and Softmax Functions*.

In [15]:
def logsumexp(logits,correct):
    
    max = torch.max(logits)
    shifted_logits = logits - max
    exp = torch.exp(shifted_logits)
    sum = torch.sum(exp)
    p = torch.log(sum)
    
    lse = max + p
    cross_entropy = lse - logits[correct]
    
    return [lse, cross_entropy]

In [1]:
import torch

In [16]:
logits_1 = torch.tensor([1000,1000])
logits_2 = torch.tensor([0,-1000])
logits_3 = torch.tensor([-1000,-1000])

logsumexp(logits_1,1), logsumexp(logits_2,1), logsumexp(logits_3,1)

([tensor(1000.6932), tensor(0.6932)],
 [tensor(0.), tensor(1000.)],
 [tensor(-999.3068), tensor(0.6932)])